# 🔧 Task 2: Data Preprocessing Pipeline

---

## 🎯 Objective
Build a robust preprocessing pipeline to clean, normalize, and prepare our UAV telemetry data for machine learning models.

## 📚 Learning Objectives
1. Understand data cleaning techniques and why they matter
2. Master normalization/standardization mathematics
3. Learn feature encoding strategies
4. Build reusable preprocessing functions (for serving consistency - Rule #29)

---

## 🔬 Mathematical Foundation: Data Preprocessing

### 1. Why Normalize Data?

Machine learning algorithms (especially gradient-based) perform better when features are on similar scales:

- **Gradient Descent Convergence**: Features with larger ranges dominate the cost function
- **Distance-Based Algorithms**: Features with larger magnitudes have more influence
- **Numerical Stability**: Prevents overflow/underflow in computations

---

### 2. Min-Max Normalization (Scaling to [0, 1])

Transforms features to a fixed range:

$$x_{normalized} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

**Properties:**
- Output range: [0, 1]
- Preserves zero values if min = 0
- Sensitive to outliers

**Inverse Transform:**
$$x_{original} = x_{normalized} \times (x_{max} - x_{min}) + x_{min}$$

---

### 3. Z-Score Standardization (Standard Scaling)

Transforms to zero mean and unit variance:

$$z = \frac{x - \mu}{\sigma}$$

Where:
- $\mu = \frac{1}{n}\sum_{i=1}^{n} x_i$ (sample mean)
- $\sigma = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \mu)^2}$ (sample standard deviation)

**Properties:**
- Output: mean = 0, std = 1
- Less sensitive to outliers than Min-Max
- Useful when features follow Gaussian distribution

**Inverse Transform:**
$$x_{original} = z \times \sigma + \mu$$

---

### 4. Robust Scaling (Median-Based)

Uses median and IQR instead of mean and std:

$$x_{robust} = \frac{x - median(x)}{IQR(x)}$$

Where: $IQR = Q_3 - Q_1$ (75th percentile - 25th percentile)

**Properties:**
- Robust to outliers
- Good for skewed distributions

---

### 5. One-Hot Encoding (Categorical Variables)

Converts categorical variable with $k$ categories to $k$ binary columns:

$$\text{terrain} \in \{clear, windy, dusty\} \rightarrow \begin{bmatrix} is\_clear \\ is\_windy \\ is\_dusty \end{bmatrix}$$

Example: "windy" → [0, 1, 0]

---

### 6. Handling Missing Values

Common strategies:
- **Mean/Median Imputation**: $x_{missing} = \mu$ or $median$
- **Forward Fill**: $x_t = x_{t-1}$ (for time series)
- **Interpolation**: Linear or polynomial

---

## 🔄 Preprocessing Workflow

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Load Raw Data  │ ──▶ │  Handle Missing │ ──▶ │ Remove Outliers │
│                 │     │     Values      │     │                 │
└─────────────────┘     └─────────────────┘     └─────────────────┘
                                                        │
                                                        ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Save Processed │ ◀── │  Encode         │ ◀── │  Normalize      │
│     Data        │     │  Categoricals   │     │  Numerics       │
└─────────────────┘     └─────────────────┘     └─────────────────┘
```

## 📦 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")

## 📥 Step 1: Load Raw Data

In [ ]:
# Load the raw data generated in Task 1
raw_data_path = '../data/raw/uav_telemetry.csv'
df = pd.read_csv(raw_data_path)

# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"✅ Data loaded successfully!")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

## 🔍 Step 2: Data Quality Check

In [ ]:
def check_data_quality(df):
    """
    Comprehensive data quality assessment.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataset
    
    Returns:
    --------
    quality_report : dict
        Dictionary containing quality metrics
    """
    quality_report = {
        'total_rows': len(df),
        'total_columns': len(df.columns),
        'missing_values': {},
        'duplicates': df.duplicated().sum(),
        'data_types': df.dtypes.to_dict(),
        'numeric_stats': {},
        'categorical_stats': {}
    }
    
    # Check missing values
    for col in df.columns:
        missing = df[col].isnull().sum()
        if missing > 0:
            quality_report['missing_values'][col] = {
                'count': missing,
                'percentage': round(missing / len(df) * 100, 2)
            }
    
    # Numeric column stats
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        quality_report['numeric_stats'][col] = {
            'min': df[col].min(),
            'max': df[col].max(),
            'mean': round(df[col].mean(), 2),
            'std': round(df[col].std(), 2),
            'zeros': (df[col] == 0).sum(),
            'negative': (df[col] < 0).sum()
        }
    
    # Categorical column stats
    cat_cols = df.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if col not in ['timestamp', 'flight_id']:
            quality_report['categorical_stats'][col] = df[col].value_counts().to_dict()
    
    return quality_report


# Run quality check
quality_report = check_data_quality(df)

print("=" * 60)
print("                  DATA QUALITY REPORT")
print("=" * 60)
print(f"\nTotal Rows: {quality_report['total_rows']}")
print(f"Total Columns: {quality_report['total_columns']}")
print(f"Duplicate Rows: {quality_report['duplicates']}")

print(f"\n📊 Missing Values:")
if quality_report['missing_values']:
    for col, stats in quality_report['missing_values'].items():
        print(f"   - {col}: {stats['count']} ({stats['percentage']}%)")
else:
    print("   ✅ No missing values found!")

print(f"\n📈 Categorical Distribution:")
for col, counts in quality_report['categorical_stats'].items():
    print(f"   {col}: {counts}")

## 🧹 Step 3: Data Cleaning - Handle Missing Values

In [ ]:
def handle_missing_values(df, strategy='median'):
    """
    Handle missing values in the dataset.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataset
    strategy : str
        'mean', 'median', or 'drop'
    
    Returns:
    --------
    df_clean : pandas.DataFrame
        Cleaned dataset
    impute_values : dict
        Values used for imputation (for serving consistency)
    """
    df_clean = df.copy()
    impute_values = {}
    
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            if strategy == 'mean':
                fill_value = df_clean[col].mean()
            elif strategy == 'median':
                fill_value = df_clean[col].median()
            else:  # drop
                df_clean = df_clean.dropna(subset=[col])
                continue
            
            df_clean[col].fillna(fill_value, inplace=True)
            impute_values[col] = fill_value
            print(f"   Imputed {col} with {strategy}: {fill_value:.2f}")
    
    # Handle categorical missing (if any)
    cat_cols = df_clean.select_dtypes(include=['object']).columns
    for col in cat_cols:
        if df_clean[col].isnull().sum() > 0:
            mode_value = df_clean[col].mode()[0]
            df_clean[col].fillna(mode_value, inplace=True)
            impute_values[col] = mode_value
    
    return df_clean, impute_values


# Apply missing value handling
print("Handling missing values...")
df_clean, impute_values = handle_missing_values(df, strategy='median')

if not impute_values:
    print("✅ No missing values to handle!")
else:
    print(f"\n✅ Imputation complete. Values saved for serving.")

## 📊 Step 4: Outlier Detection and Handling

In [ ]:
def detect_outliers_iqr(df, columns, threshold=1.5):
    """
    Detect outliers using the IQR method.
    
    Mathematical Definition:
    - Outlier if x < Q1 - threshold * IQR OR x > Q3 + threshold * IQR
    - IQR = Q3 - Q1
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataset
    columns : list
        Columns to check for outliers
    threshold : float
        IQR multiplier (1.5 standard, 3.0 for extreme outliers)
    
    Returns:
    --------
    outlier_info : dict
        Outlier statistics per column
    outlier_mask : pandas.Series
        Boolean mask for outlier rows
    """
    outlier_info = {}
    outlier_mask = pd.Series([False] * len(df), index=df.index)
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - threshold * IQR
        upper_bound = Q3 + threshold * IQR
        
        col_outliers = (df[col] < lower_bound) | (df[col] > upper_bound)
        outlier_count = col_outliers.sum()
        
        if outlier_count > 0:
            outlier_info[col] = {
                'count': outlier_count,
                'percentage': round(outlier_count / len(df) * 100, 2),
                'lower_bound': round(lower_bound, 2),
                'upper_bound': round(upper_bound, 2),
                'Q1': round(Q1, 2),
                'Q3': round(Q3, 2),
                'IQR': round(IQR, 2)
            }
        
        outlier_mask = outlier_mask | col_outliers
    
    return outlier_info, outlier_mask


def cap_outliers(df, columns, threshold=1.5):
    """
    Cap outliers at the IQR bounds (winsorization).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataset
    columns : list
        Columns to cap
    threshold : float
        IQR multiplier
    
    Returns:
    --------
    df_capped : pandas.DataFrame
        Dataset with capped outliers
    bounds : dict
        Lower and upper bounds per column
    """
    df_capped = df.copy()
    bounds = {}
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - threshold * IQR
        upper = Q3 + threshold * IQR
        
        # Cap values
        df_capped[col] = df_capped[col].clip(lower=lower, upper=upper)
        
        bounds[col] = {'lower': lower, 'upper': upper}
    
    return df_capped, bounds


# Columns to check for outliers
numeric_features = ['battery_soc', 'voltage', 'current', 'battery_temperature',
                    'wind_speed', 'dust_level', 'ambient_temperature', 'altitude',
                    'payload_weight', 'flight_speed', 'power_consumption', 'max_range']

# Detect outliers
print("Detecting outliers using IQR method (threshold = 1.5)...\n")
outlier_info, outlier_mask = detect_outliers_iqr(df_clean, numeric_features)

print("=" * 60)
print("                  OUTLIER DETECTION REPORT")
print("=" * 60)
total_outlier_rows = outlier_mask.sum()
print(f"\nTotal rows with outliers: {total_outlier_rows} ({total_outlier_rows/len(df_clean)*100:.2f}%)")

if outlier_info:
    print("\nOutliers by column:")
    for col, info in outlier_info.items():
        print(f"\n   {col}:")
        print(f"      Count: {info['count']} ({info['percentage']}%)")
        print(f"      Bounds: [{info['lower_bound']}, {info['upper_bound']}]")

In [ ]:
# Visualize outliers with box plots
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(numeric_features):
    axes[i].boxplot(df_clean[col].dropna())
    axes[i].set_title(col, fontsize=10)
    axes[i].tick_params(axis='x', which='both', bottom=False, labelbottom=False)

plt.suptitle('Box Plots for Outlier Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/outlier_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Outlier visualization saved!")

In [ ]:
# Cap outliers instead of removing (preserves data)
print("Capping outliers at IQR bounds...")
df_capped, outlier_bounds = cap_outliers(df_clean, numeric_features)

print("✅ Outliers capped successfully!")
print(f"\nBounds saved for {len(outlier_bounds)} columns.")

## 📏 Step 5: Feature Normalization (From Scratch!)

Implementing normalization functions using only NumPy:

In [ ]:
class MinMaxScaler:
    """
    Min-Max Normalization: Scale features to [0, 1] range.
    
    Formula: x_norm = (x - x_min) / (x_max - x_min)
    
    Implements fit/transform pattern for consistency between training and serving.
    """
    
    def __init__(self):
        self.min_vals = None
        self.max_vals = None
        self.feature_names = None
    
    def fit(self, X, feature_names=None):
        """
        Compute min and max values for each feature.
        
        Parameters:
        -----------
        X : numpy.ndarray
            Training data (n_samples, n_features)
        feature_names : list
            Optional feature names
        """
        X = np.array(X)
        self.min_vals = np.min(X, axis=0)
        self.max_vals = np.max(X, axis=0)
        self.feature_names = feature_names
        
        # Avoid division by zero
        self.range_vals = self.max_vals - self.min_vals
        self.range_vals[self.range_vals == 0] = 1
        
        return self
    
    def transform(self, X):
        """
        Apply Min-Max normalization.
        
        Parameters:
        -----------
        X : numpy.ndarray
            Data to transform
        
        Returns:
        --------
        X_normalized : numpy.ndarray
            Normalized data
        """
        X = np.array(X)
        return (X - self.min_vals) / self.range_vals
    
    def fit_transform(self, X, feature_names=None):
        """Fit and transform in one step."""
        self.fit(X, feature_names)
        return self.transform(X)
    
    def inverse_transform(self, X_normalized):
        """
        Convert normalized values back to original scale.
        
        Formula: x = x_norm * (x_max - x_min) + x_min
        """
        X_normalized = np.array(X_normalized)
        return X_normalized * self.range_vals + self.min_vals
    
    def get_params(self):
        """Get scaler parameters for saving."""
        return {
            'min_vals': self.min_vals.tolist(),
            'max_vals': self.max_vals.tolist(),
            'feature_names': self.feature_names
        }


class StandardScaler:
    """
    Z-Score Standardization: Transform to zero mean and unit variance.
    
    Formula: z = (x - μ) / σ
    
    Where:
    - μ = mean of feature
    - σ = standard deviation of feature
    """
    
    def __init__(self):
        self.mean_vals = None
        self.std_vals = None
        self.feature_names = None
    
    def fit(self, X, feature_names=None):
        """
        Compute mean and standard deviation for each feature.
        """
        X = np.array(X)
        self.mean_vals = np.mean(X, axis=0)
        self.std_vals = np.std(X, axis=0)
        self.feature_names = feature_names
        
        # Avoid division by zero
        self.std_vals[self.std_vals == 0] = 1
        
        return self
    
    def transform(self, X):
        """
        Apply Z-score standardization.
        """
        X = np.array(X)
        return (X - self.mean_vals) / self.std_vals
    
    def fit_transform(self, X, feature_names=None):
        """Fit and transform in one step."""
        self.fit(X, feature_names)
        return self.transform(X)
    
    def inverse_transform(self, X_standardized):
        """
        Convert standardized values back to original scale.
        
        Formula: x = z * σ + μ
        """
        X_standardized = np.array(X_standardized)
        return X_standardized * self.std_vals + self.mean_vals
    
    def get_params(self):
        """Get scaler parameters for saving."""
        return {
            'mean_vals': self.mean_vals.tolist(),
            'std_vals': self.std_vals.tolist(),
            'feature_names': self.feature_names
        }


print("✅ Scaler classes defined (from scratch using NumPy!)")

In [ ]:
# Demonstrate both scalers
sample_data = df_capped[['battery_soc', 'voltage', 'current']].values[:10]

print("Original Data (first 3 features, 10 samples):")
print(sample_data[:5])

# Min-Max scaling
mm_scaler = MinMaxScaler()
mm_scaled = mm_scaler.fit_transform(sample_data)
print("\nMin-Max Scaled (range 0-1):")
print(mm_scaled[:5].round(4))

# Standard scaling
std_scaler = StandardScaler()
std_scaled = std_scaler.fit_transform(sample_data)
print("\nStandard Scaled (mean=0, std=1):")
print(std_scaled[:5].round(4))

# Verify inverse transform
reconstructed = mm_scaler.inverse_transform(mm_scaled)
print("\nReconstructed from Min-Max (should match original):")
print(reconstructed[:5].round(2))

## 🏷️ Step 6: Encode Categorical Variables

In [ ]:
class OneHotEncoder:
    """
    One-Hot Encoding for categorical variables.
    
    Converts categorical variable with k categories to k binary columns.
    Example: 'terrain' → ['terrain_clear', 'terrain_windy', 'terrain_dusty']
    """
    
    def __init__(self):
        self.categories = None
        self.column_name = None
    
    def fit(self, X, column_name='category'):
        """
        Learn the unique categories.
        
        Parameters:
        -----------
        X : array-like
            Categorical values
        column_name : str
            Name for the encoded columns
        """
        self.categories = sorted(list(set(X)))
        self.column_name = column_name
        return self
    
    def transform(self, X):
        """
        Apply one-hot encoding.
        
        Returns:
        --------
        encoded : numpy.ndarray
            Binary encoded matrix (n_samples, n_categories)
        column_names : list
            Names for each encoded column
        """
        n_samples = len(X)
        n_categories = len(self.categories)
        
        # Create binary matrix
        encoded = np.zeros((n_samples, n_categories), dtype=int)
        
        for i, val in enumerate(X):
            if val in self.categories:
                cat_idx = self.categories.index(val)
                encoded[i, cat_idx] = 1
        
        column_names = [f"{self.column_name}_{cat}" for cat in self.categories]
        
        return encoded, column_names
    
    def fit_transform(self, X, column_name='category'):
        """Fit and transform in one step."""
        self.fit(X, column_name)
        return self.transform(X)
    
    def get_params(self):
        """Get encoder parameters for saving."""
        return {
            'categories': self.categories,
            'column_name': self.column_name
        }


# Encode terrain_type
encoder = OneHotEncoder()
terrain_encoded, terrain_cols = encoder.fit_transform(df_capped['terrain_type'], 'terrain')

print("One-Hot Encoding for 'terrain_type':")
print(f"\nCategories: {encoder.categories}")
print(f"Encoded columns: {terrain_cols}")
print(f"\nSample encoding (first 5 rows):")
print(pd.DataFrame(terrain_encoded[:5], columns=terrain_cols))

## 🔄 Step 7: Build Complete Preprocessing Pipeline

In [ ]:
class PreprocessingPipeline:
    """
    Complete preprocessing pipeline that can be saved and reused.
    
    This ensures consistency between training and serving (Rule #29, #32).
    """
    
    def __init__(self):
        self.numeric_scaler = StandardScaler()
        self.target_scaler = MinMaxScaler()
        self.encoders = {}
        self.feature_columns = None
        self.target_columns = None
        self.outlier_bounds = None
    
    def fit(self, df, numeric_features, categorical_features, target_features):
        """
        Fit all transformers on training data.
        
        Parameters:
        -----------
        df : pandas.DataFrame
            Training data
        numeric_features : list
            Numeric feature column names
        categorical_features : list
            Categorical column names
        target_features : list
            Target column names
        """
        self.feature_columns = numeric_features
        self.target_columns = target_features
        
        # Fit numeric scaler
        self.numeric_scaler.fit(df[numeric_features].values, numeric_features)
        
        # Fit target scaler
        self.target_scaler.fit(df[target_features].values, target_features)
        
        # Fit categorical encoders
        for col in categorical_features:
            encoder = OneHotEncoder()
            encoder.fit(df[col], col)
            self.encoders[col] = encoder
        
        return self
    
    def transform(self, df):
        """
        Transform data using fitted transformers.
        
        Returns:
        --------
        X : numpy.ndarray
            Transformed features
        y : numpy.ndarray
            Transformed targets
        feature_names : list
            List of all feature names
        """
        # Scale numeric features
        X_numeric = self.numeric_scaler.transform(df[self.feature_columns].values)
        
        # Encode categorical features
        X_categorical = []
        cat_feature_names = []
        for col, encoder in self.encoders.items():
            encoded, col_names = encoder.transform(df[col])
            X_categorical.append(encoded)
            cat_feature_names.extend(col_names)
        
        # Combine features
        if X_categorical:
            X_categorical = np.hstack(X_categorical)
            X = np.hstack([X_numeric, X_categorical])
            feature_names = list(self.feature_columns) + cat_feature_names
        else:
            X = X_numeric
            feature_names = list(self.feature_columns)
        
        # Scale targets
        y = self.target_scaler.transform(df[self.target_columns].values)
        
        return X, y, feature_names
    
    def fit_transform(self, df, numeric_features, categorical_features, target_features):
        """Fit and transform in one step."""
        self.fit(df, numeric_features, categorical_features, target_features)
        return self.transform(df)
    
    def inverse_transform_target(self, y_scaled):
        """Convert scaled targets back to original scale."""
        return self.target_scaler.inverse_transform(y_scaled)
    
    def save_params(self, filepath):
        """Save all parameters for serving."""
        params = {
            'numeric_scaler': self.numeric_scaler.get_params(),
            'target_scaler': self.target_scaler.get_params(),
            'encoders': {col: enc.get_params() for col, enc in self.encoders.items()},
            'feature_columns': self.feature_columns,
            'target_columns': self.target_columns
        }
        with open(filepath, 'w') as f:
            json.dump(params, f, indent=2)
        print(f"✅ Pipeline parameters saved to: {filepath}")


print("✅ PreprocessingPipeline class defined!")

## 🚀 Step 8: Apply Pipeline to Data

In [ ]:
# Define feature groups
numeric_features = [
    'battery_soc', 'voltage', 'current', 'battery_temperature',
    'wind_speed', 'dust_level', 'ambient_temperature',
    'altitude', 'payload_weight', 'flight_speed',
    'planned_distance', 'power_consumption'
]

categorical_features = ['terrain_type']

target_features = ['max_range']  # Regression target

# Create and apply pipeline
pipeline = PreprocessingPipeline()
X, y, feature_names = pipeline.fit_transform(
    df_capped,
    numeric_features,
    categorical_features,
    target_features
)

print("=" * 60)
print("              PREPROCESSING COMPLETE")
print("=" * 60)
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape: {y.shape}")
print(f"\nFeature names ({len(feature_names)}):")
for i, name in enumerate(feature_names):
    print(f"   {i+1}. {name}")

In [ ]:
# Verify scaling worked correctly
print("\nScaling Verification:")
print("\nNumeric features (standardized - should have mean≈0, std≈1):")
for i, col in enumerate(numeric_features[:5]):
    print(f"   {col}: mean={X[:, i].mean():.4f}, std={X[:, i].std():.4f}")

print("\nCategorical features (one-hot - should be 0 or 1):")
cat_start = len(numeric_features)
for i, name in enumerate(feature_names[cat_start:]):
    unique_vals = np.unique(X[:, cat_start + i])
    print(f"   {name}: unique values = {unique_vals}")

## 📊 Step 9: Visualize Preprocessing Effects

In [ ]:
# Before vs After normalization
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

sample_features = ['battery_soc', 'voltage', 'power_consumption']

for i, col in enumerate(sample_features):
    col_idx = numeric_features.index(col)
    
    # Before (original)
    axes[0, i].hist(df_capped[col], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
    axes[0, i].set_title(f'{col} (Original)', fontsize=11)
    axes[0, i].set_xlabel('Value')
    axes[0, i].set_ylabel('Frequency')
    
    # After (normalized)
    axes[1, i].hist(X[:, col_idx], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[1, i].set_title(f'{col} (Standardized)', fontsize=11)
    axes[1, i].set_xlabel('Standardized Value')
    axes[1, i].set_ylabel('Frequency')

plt.suptitle('Before vs After Standardization', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/normalization_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Normalization comparison visualization saved!")

## 💾 Step 10: Save Processed Data

In [ ]:
# Create processed dataframe
df_processed = pd.DataFrame(X, columns=feature_names)
df_processed['max_range_scaled'] = y.flatten()
df_processed['max_range_original'] = df_capped['max_range'].values
df_processed['mission_success'] = df_capped['mission_success'].values
df_processed['flight_id'] = df_capped['flight_id'].values
df_processed['timestamp'] = df_capped['timestamp'].values

# Save processed data
processed_path = '../data/processed/uav_telemetry_processed.csv'
df_processed.to_csv(processed_path, index=False)

# Save pipeline parameters
pipeline.save_params('../models/preprocessing_params.json')

print(f"\n✅ Processed data saved to: {processed_path}")
print(f"   Shape: {df_processed.shape}")
print(f"\nProcessed DataFrame preview:")
df_processed.head()

## 📊 Step 11: Train/Test Split (For Later Tasks)

In [ ]:
def train_test_split(X, y, test_size=0.2, random_state=42):
    """
    Split data into training and test sets.
    
    Implements random shuffling with reproducible results.
    
    Parameters:
    -----------
    X : numpy.ndarray
        Feature matrix
    y : numpy.ndarray
        Target vector
    test_size : float
        Proportion of data for testing (0-1)
    random_state : int
        Random seed for reproducibility
    
    Returns:
    --------
    X_train, X_test, y_train, y_test : numpy.ndarray
        Split datasets
    """
    np.random.seed(random_state)
    
    n_samples = len(X)
    n_test = int(n_samples * test_size)
    
    # Shuffle indices
    indices = np.arange(n_samples)
    np.random.shuffle(indices)
    
    # Split
    test_indices = indices[:n_test]
    train_indices = indices[n_test:]
    
    X_train = X[train_indices]
    X_test = X[test_indices]
    y_train = y[train_indices]
    y_test = y[test_indices]
    
    return X_train, X_test, y_train, y_test


# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

print("Train/Test Split Complete:")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_test shape: {X_test.shape}")
print(f"   y_train shape: {y_train.shape}")
print(f"   y_test shape: {y_test.shape}")

# Save splits
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/X_test.npy', X_test)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy', y_test)

print("\n✅ Train/test splits saved as .npy files!")

---

## 📚 Learning Resources

### Blogs & Articles
1. **Feature Scaling**: [Why, How and When to Scale Your Features](https://towardsdatascience.com/all-about-feature-scaling-bcc0ad75cb35)
2. **Handling Outliers**: [Outlier Detection Methods](https://www.analyticsvidhya.com/blog/2021/05/detecting-and-treating-outliers-treating-the-odd-one-out/)
3. **One-Hot Encoding**: [Encoding Categorical Data](https://machinelearningmastery.com/one-hot-encoding-for-categorical-data/)

### YouTube Videos
1. **Data Preprocessing**: [Complete Data Preprocessing Tutorial](https://www.youtube.com/watch?v=gxX4ELp4Yrw)
2. **Normalization vs Standardization**: [When to Use Which](https://www.youtube.com/watch?v=mnKm3YP56PY)

### Research Papers
1. "On the Importance of Feature Scaling in Machine Learning" - Various studies on gradient descent convergence
2. Zinkevich, M. (2017). "Rules of Machine Learning" - Google's ML Engineering Best Practices

---

## ✅ Task 2 Complete!

### What We Accomplished:
- ✅ Explained normalization mathematics (Min-Max, Z-Score, Robust)
- ✅ Built custom scalers from scratch using NumPy
- ✅ Implemented one-hot encoding for categorical variables
- ✅ Created reusable PreprocessingPipeline class
- ✅ Handled outliers with IQR method
- ✅ Saved processed data and pipeline parameters
- ✅ Created train/test split

### Key Files Created:
- `data/processed/uav_telemetry_processed.csv`
- `data/processed/X_train.npy`, `X_test.npy`, `y_train.npy`, `y_test.npy`
- `models/preprocessing_params.json`

### Next Task: Exploratory Data Analysis (EDA)
Deep dive into data patterns with comprehensive visualizations!

---